# Modelo de Machine Learning - Predição de Classes de Demanda
Este notebook treina um classificador para prever a probabilidade da demanda ser Baixa, Média ou Alta para a categoria 'Temperos & Condimentos'.

In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. CARGA E FILTRAGEM
df = pd.read_csv('data/database_final.csv', low_memory=False)
df['DATA_ATEND'] = pd.to_datetime(df['DATA_ATEND'])

cat_alvo = 'Temperos & Condimentos'
filial_alvo = 'SHOPPING'
df_filtrado = df[(df['CATEGORIA'] == cat_alvo) & (df['FILIAL'] == filial_alvo)].copy()

# Agregação diária
df_diario = df_filtrado.groupby('DATA_ATEND').agg(DEMANDA=('FATUR_VENDA', 'sum')).reset_index()
df_diario = df_diario.sort_values('DATA_ATEND')

print(f"Dataset carregado: {len(df_diario)} dias de histórico.")

Dataset carregado: 362 dias de histórico.


## 2. Rotulação (Target Definition)
Definimos as classes 'Baixa', 'Média' e 'Alta' com base no comportamento histórico (quantis).

In [3]:
q33 = df_diario['DEMANDA'].quantile(0.33)
q66 = df_diario['DEMANDA'].quantile(0.66)

def rotular_demanda(valor):
    if valor <= q33: return 'Baixa'
    if valor <= q66: return 'Média'
    return 'Alta'

df_diario['CLASSE'] = df_diario['DEMANDA'].apply(rotular_demanda)

print("Distribuição das classes:")
print(df_diario['CLASSE'].value_counts())
df_diario.head()

Distribuição das classes:
CLASSE
Alta     123
Baixa    120
Média    119
Name: count, dtype: int64


,DATA_ATEND,DEMANDA,CLASSE
0,2024-01-02,933.93,Baixa
1,2024-01-03,1131.89,Baixa
2,2024-01-04,1168.05,Baixa
3,2024-01-05,1474.01,Média
4,2024-01-06,1625.33,Alta


## 3. Engenharia de Atributos e Pipeline de Features
Criamos uma função que transforma uma data em um vetor de características que o modelo entende.

In [4]:
def extract_features(df_input):
    temp_df = df_input.copy()
    temp_df['MES'] = temp_df['DATA_ATEND'].dt.month
    temp_df['DIA_SEMANA'] = temp_df['DATA_ATEND'].dt.dayofweek
    temp_df['EH_FIM_DE_SEMANA'] = temp_df['DIA_SEMANA'].isin([5, 6]).astype(int)
    
    # Estações (Simplificado)
    def get_est_num(data):
        m = data.month
        if m in [1, 2, 3]: return 1 # Verao
        if m in [4, 5, 6]: return 2 # Outono
        if m in [7, 8, 9]: return 3 # Inverno
        return 4 # Primavera
    
    temp_df['ESTACAO_NUM'] = temp_df['DATA_ATEND'].apply(get_est_num)
    
    # Selecionamos as colunas numéricas finais
    features = ['MES', 'DIA_SEMANA', 'EH_FIM_DE_SEMANA', 'ESTACAO_NUM']
    return temp_df[features]

X = extract_features(df_diario)
y = df_diario['CLASSE']

## 4. Divisão Temporal (Train / Test)
Como se trata de dados no tempo, usamos os dados mais recentes para teste.

In [5]:
split_idx = int(len(X) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Treino: {len(X_train)} dias | Teste: {len(X_test)} dias")

Treino: 289 dias | Teste: 73 dias


## 5. Treinamento do Modelo
Usaremos Random Forest, que é excelente para lidar com variáveis de calendário.

In [6]:
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

# Avaliação
y_pred = model.predict(X_test)
print("=== Relatório de Classificação (Teste) ===")
print(classification_report(y_test, y_pred))

=== Relatório de Classificação (Teste) ===
              precision    recall  f1-score   support

        Alta       0.48      0.93      0.63        27
       Baixa       0.73      0.33      0.46        24
       Média       0.50      0.23      0.31        22

    accuracy                           0.52        73
   macro avg       0.57      0.50      0.47        73
weighted avg       0.57      0.52      0.48        73



## 6. Interface de Predição Probabilística
Esta função é o coração da Fase 3: ela retorna as probabilidades para alimentar a Árvore de Decisão.

In [9]:
def prever_probabilidades(dia, mes, ano):
    data_str = f"{ano}-{mes:02d}-{dia:02d}"
    data_dt = pd.to_datetime([data_str])
    
    # Prepara as features
    input_df = pd.DataFrame({'DATA_ATEND': data_dt})
    X_input = extract_features(input_df)
    
    # Predição de probabilidades
    probs = model.predict_proba(X_input)[0]
    classes = model.classes_
    
    res = {classes[i]: probs[i] for i in range(len(classes))}
    
    print(f"\nProbabilidades de Demanda para {data_str}:")
    for classe, prob in res.items():
        print(f"- {classe}: {prob*100:.2f}%")
    
    return res

# Exemplo de uso:
prever_probabilidades(25, 12, 2025)


Probabilidades de Demanda para 2025-12-25:
- Alta: 39.64%
- Baixa: 0.25%
- Média: 60.11%


{'Alta': np.float64(0.39635652054557474),
 'Baixa': np.float64(0.0025129533678756475),
 'Média': np.float64(0.6011305260865496)}